# 🎯 Ridge & Lasso (Regularization)
**ISLP Chapter 6 · Pattern Recognition for the Rest of Us**

> When you have many predictors, OLS overfits. Ridge and Lasso add a penalty that shrinks coefficients toward zero — trading a small increase in bias for a big reduction in variance.

### What you'll learn
- Why OLS fails with many or correlated predictors
- Ridge regression (L2 penalty) — shrinks all coefficients
- Lasso regression (L1 penalty) — shrinks *and* zeros out coefficients
- How to choose λ using cross-validation
- When to use Ridge vs Lasso

### Dataset: Hitters (baseball salary prediction, 20 predictors)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor':'white','axes.facecolor':'#f8f9fa',
    'axes.grid':True,'grid.alpha':0.4,
    'axes.spines.top':False,'axes.spines.right':False,'font.size':11
})
print("✓ Setup complete")
from sklearn.linear_model import Ridge, Lasso, RidgeCV, LassoCV, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import Pipeline

In [ ]:
# ── Load datasets from the course repo ──────────────────────────────────────
import subprocess, os

# Clone the course repo if not already present (works in Colab)
if not os.path.exists('pattern-recognition-notebooks'):
    subprocess.run(['git','clone','--depth','1',
                    'https://github.com/ladataanalytics/pattern-recognition-notebooks.git'],
                   capture_output=True)

DATA_DIR = 'pattern-recognition-notebooks/data'
if not os.path.exists(DATA_DIR):
    # Fallback: already in repo root (e.g. running locally)
    DATA_DIR = '../data'

print(f"✓ Data directory: {DATA_DIR}")
print(f"✓ Available datasets: {os.listdir(DATA_DIR)}")

### Load Dataset: Hitters

In [ ]:
import pandas as pd, numpy as np
hitters = pd.read_csv(f'{DATA_DIR}/Hitters.csv').dropna()
hitters = pd.get_dummies(hitters, drop_first=True, dtype=float)
X = hitters.drop('Salary', axis=1)
y = np.log(hitters['Salary'])
print(f"Hitters: {X.shape}  |  Predicting: log(Salary)")

## 🎯 Part 1 — The Problem: OLS With Many Predictors

With 20 predictors, OLS fits *perfectly* on training data but generalizes poorly. When predictors are correlated, OLS coefficients become unstable — small changes in data cause large swings in estimates.

**Regularization** adds a penalty term to the OLS objective:

**Ridge (L2):** minimize RSS + λ × Σβⱼ²  
**Lasso (L1):** minimize RSS + λ × Σ|βⱼ|

λ (lambda) controls penalty strength:
- λ=0 → same as OLS  
- λ→∞ → all coefficients → 0

In [ ]:
# Show coefficient paths as lambda varies
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42)
scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_te_s  = scaler.transform(X_te)

alphas = np.logspace(-3, 5, 200)
ridge_coefs = []
lasso_coefs = []

for a in alphas:
    ridge_coefs.append(Ridge(alpha=a).fit(X_tr_s, y_tr).coef_)
    try:
        lasso_coefs.append(Lasso(alpha=a, max_iter=10000).fit(X_tr_s, y_tr).coef_)
    except:
        lasso_coefs.append(np.zeros(X_tr_s.shape[1]))

ridge_coefs = np.array(ridge_coefs)
lasso_coefs = np.array(lasso_coefs)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i in range(ridge_coefs.shape[1]):
    axes[0].plot(np.log10(alphas), ridge_coefs[:,i], lw=0.8, alpha=0.7)
axes[0].axvline(0, color='black', lw=0.8, ls='--', alpha=0.5)
axes[0].set_xlabel('log₁₀(λ)')
axes[0].set_ylabel('Standardized Coefficient')
axes[0].set_title('Ridge — coefficients shrink smoothly\n(none reach exactly zero)')

for i in range(lasso_coefs.shape[1]):
    axes[1].plot(np.log10(alphas), lasso_coefs[:,i], lw=0.8, alpha=0.7)
axes[1].axvline(0, color='black', lw=0.8, ls='--', alpha=0.5)
axes[1].set_xlabel('log₁₀(λ)')
axes[1].set_ylabel('Standardized Coefficient')
axes[1].set_title('Lasso — coefficients reach exactly zero\n(automatic feature selection!)')

plt.suptitle('Coefficient Paths: Ridge vs Lasso', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()
print("\n📌 Key difference: Lasso produces sparse solutions (some β=0 exactly).")
print("   Ridge keeps all predictors but shrinks them.")

In [ ]:
# Choose optimal lambda via cross-validation
ridge_cv = RidgeCV(alphas=np.logspace(-3, 5, 100), cv=10)
lasso_cv = LassoCV(alphas=np.logspace(-4, 2, 100), cv=10, max_iter=10000)

ridge_cv.fit(X_tr_s, y_tr)
lasso_cv.fit(X_tr_s, y_tr)

print(f"Ridge optimal λ: {ridge_cv.alpha_:.4f}")
print(f"Lasso optimal λ: {lasso_cv.alpha_:.4f}")
print(f"Lasso zeroed out: {np.sum(lasso_cv.coef_ == 0)}/{X.shape[1]} predictors")

# Compare test MSE
ols   = LinearRegression().fit(X_tr_s, y_tr)
models_res = {
    'OLS':   mean_squared_error(y_te, ols.predict(X_te_s)),
    'Ridge': mean_squared_error(y_te, ridge_cv.predict(X_te_s)),
    'Lasso': mean_squared_error(y_te, lasso_cv.predict(X_te_s)),
}

print("\n=== Test MSE Comparison ===")
for name, mse in models_res.items():
    print(f"  {name:<8} MSE = {mse:.4f}  {'← best' if mse == min(models_res.values()) else ''}")

In [ ]:
# Visualize: which predictors Lasso keeps vs zeros
coef_df = pd.DataFrame({
    'Predictor': X.columns,
    'OLS': ols.coef_,
    'Ridge': ridge_cv.coef_,
    'Lasso': lasso_cv.coef_
}).sort_values('Lasso', key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(11, 5))
x_pos = np.arange(len(coef_df))
w = 0.28
ax.bar(x_pos - w, coef_df['OLS'],   w, label='OLS',   color='#888',    alpha=0.7)
ax.bar(x_pos,     coef_df['Ridge'], w, label='Ridge',  color='#1e5fa8', alpha=0.8)
ax.bar(x_pos + w, coef_df['Lasso'], w, label='Lasso',  color='#e85d20', alpha=0.8)
ax.axhline(0, color='black', lw=0.8)
ax.set_xticks(x_pos)
ax.set_xticklabels(coef_df['Predictor'], rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Standardized Coefficient')
ax.set_title('OLS vs Ridge vs Lasso Coefficients')
ax.legend()
plt.tight_layout()
plt.show()
print("\n📌 Orange bars at zero = Lasso has eliminated that predictor entirely.")

In [ ]:
# @title 📝 Quiz — Ridge & Lasso
# @markdown Answer each question in the box below, then run the AI grading cell.

# @markdown **Q1:** What penalty does Ridge add to the OLS objective?
# @markdown **Q2:** Why does Lasso produce exactly-zero coefficients but Ridge does not?
# @markdown **Q3:** How do you choose the optimal λ in practice?
# @markdown **Q4:** When would you prefer Ridge over Lasso?
# @markdown **Q5:** What happens to all coefficients as λ → ∞?

q1 = "" # @param {type:"string", placeholder:"your answer"}
q2 = "" # @param {type:"string", placeholder:"your answer"}
q3 = "" # @param {type:"string", placeholder:"your answer"}
q4 = "" # @param {type:"string", placeholder:"your answer"}
q5 = "" # @param {type:"string", placeholder:"your answer"}

# Collect into answers dict for grading cell
answers = {"q1": q1, "q2": q2, "q3": q3, "q4": q4, "q5": q5}
missing = [k for k,v in answers.items() if not str(v).strip()]
print(f"  {5-len(missing)}/5 answered — run the AI grading cell below for feedback")

### 📤 Submit your results

In [ ]:
_NB_NAME_ = "Ridge & Lasso"
# @title 🤖 AI Feedback — enter your username and click ▶ Run
# @markdown **No API key needed.** AI grading runs free inside your Colab session.
# @markdown
# @markdown Enter your GitHub username, then click ▶ Run for question-by-question feedback.

GITHUB_USERNAME = "" # @param {type:"string", placeholder:"e.g. jsmith42"}

# ── runs automatically — do not edit below ───────────────────
import json, textwrap, re as _re, time
_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

def _get_quiz_questions():
    """Pull question text from the quiz cell @markdown lines."""
    try:
        import ipynbname
        nb_path = ipynbname.path()
        with open(nb_path) as f:
            nb = json.load(f)
        for cell in nb["cells"]:
            src = "".join(cell.get("source", []))
            if "@title" in src and "Quiz" in src:
                qs = re.findall(r"@markdown \*\*Q\d+:\*\* (.+)", src)
                if qs: return qs
    except Exception:
        pass
    return []

def _call_gemini(prompt, max_retries=3):
    """Call Gemini with retry on 429 rate limit."""
    last_err = None
    for attempt in range(max_retries):
        try:
            import google.generativeai as genai
            import google.auth, google.auth.transport.requests
            creds, _ = google.auth.default()
            creds.refresh(google.auth.transport.requests.Request())
            genai.configure(credentials=creds)
            model  = genai.GenerativeModel("gemini-2.0-flash")
            result = model.generate_content(
                prompt,
                generation_config={"max_output_tokens": 1500, "temperature": 0.3}
            )
            return result.text, "Gemini via Colab"
        except Exception as e:
            last_err = str(e)
            if "429" in str(e) or "quota" in str(e).lower():
                wait = 2 ** attempt
                print(f"  Rate limit hit — waiting {wait}s before retry {attempt+1}/{max_retries}...")
                time.sleep(wait)
                continue
            break
    # Try stored key as fallback
    try:
        from google.colab import userdata
        key = userdata.get("GEMINI_API_KEY")
        if key and len(key) > 10:
            import google.generativeai as genai
            genai.configure(api_key=key)
            model  = genai.GenerativeModel("gemini-2.0-flash")
            result = model.generate_content(prompt)
            return result.text, "Gemini via key"
    except Exception:
        pass
    return None, last_err

def _build_prompt(answers_dict, nb_name, questions):
    answered  = {k: v for k, v in answers_dict.items() if str(v).strip()}
    n_total   = len(answers_dict)
    n_done    = len(answered)

    # Pair each question with the student answer
    qa_pairs = ""
    for i, (k, v) in enumerate(answers_dict.items(), 1):
        q_text = questions[i-1] if i-1 < len(questions) else f"Question {i}"
        a_text = str(v).strip() if str(v).strip() else "(no answer)"
        qa_pairs += f"Q{i}: {q_text}\nA{i}: {a_text}\n\n"

    return f"""You are a TA grading a student quiz for the "{nb_name}" notebook in a data science course called "Data Pattern Recognition for the Rest of Us" (based on ISLP and business analytics).

{qa_pairs.strip()}

For EACH question:
- Decide if the answer is CORRECT, PARTIALLY CORRECT, or INCORRECT
- A short paraphrase or reasonable approximation counts as CORRECT — do not require exact wording
- For INCORRECT or PARTIAL: name the specific concept they need to review (1 sentence)
- For CORRECT: brief confirmation of what they got right (1 sentence)

Then give an overall summary.

Respond ONLY in this exact JSON format (no markdown fences, no extra text):
{{
  "questions": [
    {{
      "q": 1,
      "status": "<CORRECT|PARTIAL|INCORRECT>",
      "comment": "<one specific sentence>"
    }}
  ],
  "quiz_score": <integer 0-{n_total}>,
  "grade": "<Excellent|Good|Needs Review|Incomplete>",
  "summary": "<2 sentences overall: strengths and what to revisit>",
  "review_tip": "<one specific concept, chapter, or notebook to review if they struggled — or 'Great work!' if excellent>"
}}

Scoring guide: CORRECT=1pt, PARTIAL=0.5pt (round to nearest int), INCORRECT=0pt."""

def _show(result, username, nb_name, source, questions):
    STATUS_ICON  = {"CORRECT": "\u2705", "PARTIAL": "\u26a0\ufe0f", "INCORRECT": "\u274c"}
    STATUS_COLOR = {"CORRECT": "\033[92m", "PARTIAL": "\033[93m", "INCORRECT": "\033[91m"}
    R = "\033[0m"
    grade = result.get("grade", "?")
    GRADE_COLOR = {"Excellent":"\033[92m","Good":"\033[94m",
                   "Needs Review":"\033[93m","Incomplete":"\033[91m"}
    GC = GRADE_COLOR.get(grade, "")
    n  = len(answers)
    qs = result.get("quiz_score", 0)
    bar = "\u2588"*qs + "\u2591"*(n-qs)

    print("\n" + "\u2550"*60)
    print(f"  \U0001f916 AI Feedback \u2014 {nb_name}")
    if source: print(f"  Powered by   {source}")
    print("\u2550"*60)
    print(f"  Student  : {'@'+username if username else '\u26a0 set GITHUB_USERNAME above'}")
    print(f"  Grade    : {GC}{grade}{R}")
    print(f"  Score    : {qs}/{n}  [{bar}]")
    print()

    # Question-by-question breakdown
    q_results = result.get("questions", [])
    if q_results:
        print("  \u2500"*29)
        for qr in q_results:
            idx    = qr.get("q", 0) - 1
            status = qr.get("status", "?")
            icon   = STATUS_ICON.get(status, "\u2753")
            color  = STATUS_COLOR.get(status, "")
            comment= qr.get("comment", "")
            q_text = questions[idx] if idx < len(questions) else f"Question {qr.get('q','?')}"
            # Truncate long question text for display
            q_short = q_text[:55] + "..." if len(q_text) > 55 else q_text
            print(f"  {icon} {color}Q{qr.get('q','?')} {status}{R}")
            print(f"     {q_short}")
            if comment:
                for line in textwrap.wrap(comment, 54):
                    print(f"     \u2192 {line}")
            print()

    # Summary
    summary = result.get("summary", "")
    if summary:
        print("  \u2500"*29)
        print("  Overall:")
        for line in textwrap.wrap(summary, 56):
            print(f"  {line}")

    # Review tip
    tip = result.get("review_tip", "")
    if tip and tip != "Great work!":
        print()
        for line in textwrap.wrap(f"\U0001f4d6 Review: {tip}", 56):
            print(f"  {line}")
    elif tip == "Great work!":
        print()
        print("  \U0001f4d6 Great work! Keep going.")

    print("\u2550"*60 + "\n")

def _fallback_grade(answers_dict):
    """Smart fallback — grade on completion only, no length penalty."""
    n  = len(answers_dict)
    nd = sum(1 for v in answers_dict.values() if str(v).strip())
    if nd == 0:
        return {"quiz_score":0,"grade":"Incomplete",
                "questions":[],
                "summary":"No answers provided — fill in the quiz above.",
                "review_tip":"Complete the quiz and re-run for AI feedback."}
    elif nd == n:
        return {"quiz_score":n,"grade":"Good",
                "questions":[],
                "summary":f"All {n} questions answered. AI grading was unavailable — re-run to get question-by-question feedback.",
                "review_tip":"Re-run this cell in a few minutes for detailed AI feedback."}
    else:
        return {"quiz_score":nd,"grade":"Needs Review",
                "questions":[],
                "summary":f"{nd}/{n} questions answered. Complete the remaining {n-nd} and re-run.",
                "review_tip":"Answer all questions for full feedback."}

# ── Main ──────────────────────────────────────────────────────
if "answers" not in globals():
    print("\u26a0  Run the quiz cell above first, then re-run this cell.")
else:
    nd       = sum(1 for v in answers.values() if str(v).strip())
    username = GITHUB_USERNAME.strip()
    questions = _get_quiz_questions()

    print(f"\n  Notebook : {_NB_TITLE}  \u2022  {nd}/{len(answers)} answered")
    if username: print(f"  Student  : @{username}")
    print(f"  Grading  : please wait...\n")

    prompt     = _build_prompt(answers, _NB_TITLE, questions)
    raw, src   = _call_gemini(prompt)

    if raw:
        try:
            clean  = _re.sub(r"```(?:json)?\s*|```","",raw).strip()
            result = json.loads(clean)
        except Exception:
            nd2    = sum(1 for v in answers.values() if str(v).strip())
            result = {"quiz_score":nd2,"grade":"Good" if nd2==len(answers) else "Needs Review",
                      "questions":[],"summary":raw[:400],"review_tip":""}
    else:
        if src: print(f"  \u26a0 Gemini unavailable ({src[:60]}) \u2014 showing completion feedback\n")
        result = _fallback_grade(answers)

    _show(result, username, _NB_TITLE, src if raw else None, questions)
    print("  \U0001f4be  File \u2192 Save a copy in GitHub \u2192 your fork\n")
